In [ ]:
import pandas as pd
import numpy as np

df_tpm = pd.read_csv("../../rna-seq/star_rsem/rsem.merged.gene_tpm.tsv", sep="\t")
df_tpm.set_index("gene_id", inplace=True)
df_tpm

In [ ]:
df_genePIA = pd.read_csv("PI_100kb_Agene_id")
df_genePIA.columns = ["gene_id"]
df_genePIA.set_index("gene_id", inplace=True)
df_tpmPIA = df_tpm.loc[df_genePIA.index].drop(["transcript_id(s)", "WT_REP1", "WT_REP2", "WT_REP3"], axis=1)
df_tpmPI_A = df_tpmPIA[["TGEV_REP1","TGEV_REP2", "TGEV_REP3"]].mean(axis=1)
df_tpmPI_A

In [ ]:
df_genePIB = pd.read_csv("PI_100kb_Bgene_id")
df_genePIB.columns = ["gene_id"]
df_genePIB.set_index("gene_id", inplace=True)
df_tpmPIB = df_tpm.loc[df_genePIB.index].drop(["transcript_id(s)", "WT_REP1", "WT_REP2", "WT_REP3"], axis=1)
df_tpmPI_B = df_tpmPIB[["TGEV_REP1","TGEV_REP2", "TGEV_REP3"]].mean(axis=1)
df_tpmPI_B

In [ ]:
df = pd.concat([df_tpmPI_A, df_tpmPI_B], axis=1)
df.columns = ["PI_A", "PI_B"]
df["PI_A"].dropna()

In [ ]:
df["PI_B"].dropna()

In [ ]:
df_p = df.melt()
df_p

In [ ]:
df_p.columns = ["Type", "Tpm"]
df_p['Tpm'] = df_p['Tpm'].apply(lambda x: np.log(x + 0.00001))
df_p

In [ ]:
from scipy.stats import ranksums
z, p = ranksums(df['PI_A'].dropna(), df['PI_B'].dropna())
print(z, p)

In [ ]:
import plotly.express as px
import plotly.io as pio
import numpy as np

fig = px.box(df_p.dropna(), 
             x="Type",
             y="Tpm", 
             color="Type",
             notched=True
            )

fig.update_layout(
    width=500,
    height=500,
    paper_bgcolor='white',
    plot_bgcolor='white'
)

# Add title and set title position
fig.update_layout(
    title={
        'text': "A Vs B",
        'y': 0.9,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    }
)


fig.update_yaxes(range=[-8, 5])

# Save figure as PDF file
pio.write_image(fig, 'PIboxplot.pdf')
fig.show()